# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](https://github.com/wtraquinas/lab-neural-networks/blob/master/your-code/tttboard.jpg?raw=1)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Step 1: Read the CSV into a DataFrame
df = pd.read_csv("tic-tac-toe.csv")

# Step 2: Inspect the dataset
print("=== STEP 2: INSPECTION ===")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].unique()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTarget distribution:\n{df['class'].value_counts()}")

# Step 3: Convert categorical values to numeric
print("\n=== STEP 3: ENCODING ===")
mapping = {'x': 1, 'o': -1, 'b': 0}
feature_cols = [c for c in df.columns if c != 'class']

for col in feature_cols:
    df[col] = df[col].map(mapping)

df['class'] = df['class'].astype(int)  # True -> 1, False -> 0

print(f"After encoding (first 5 rows):\n{df.head()}")

# Step 4: Separate inputs and output
print("\n=== STEP 4: SPLIT FEATURES / TARGET ===")
X = df[feature_cols]
y = df['class']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Step 5: Normalize the input data
print("\n=== STEP 5: NORMALIZATION ===")
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print(f"X_scaled (first 5 rows):\n{X_scaled.head()}")
print(f"\nX_scaled stats:\n{X_scaled.describe().round(3)}")


=== STEP 2: INSPECTION ===
Shape: (958, 10)

First 5 rows:
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True

Data types:
TL       object
TM       object
TR       object
ML       object
MM       object
MR       object
BL       object
BM       object
BR       object
class      bool
dtype: object

Unique values per column:
  TL: ['x' 'o' 'b']
  TM: ['x' 'o' 'b']
  TR: ['x' 'o' 'b']
  ML: ['x' 'o' 'b']
  MM: ['o' 'b' 'x']
  MR: ['o' 'b' 'x']
  BL: ['x' 'o' 'b']
  BM: ['o' 'x' 'b']
  BR: ['o' 'x' 'b']
  class: [ True False]

Missing values:
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64

Target distribution:
class
True     626
False    332
Name: count, dtype: int64

=== STEP 3: ENCODING ===
After encoding (first 5 rows):
   TL  TM  TR  ML  MM  MR  BL  

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [6]:
from keras.optimizers import Adam
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras

# ── Step 1: Split into training and test sets ────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

# ── Step 2: Create a Sequential model ────────────
model = keras.Sequential()

# ── Step 3: Add layers ──────────
model.add(keras.layers.Input(shape=(X_train.shape[1],)))   # input layer
model.add(keras.layers.Dense(64, activation='relu'))        # hidden layer 1
model.add(keras.layers.Dense(32, activation='relu'))        # hidden layer 2
model.add(keras.layers.Dense(16, activation='relu'))        # hidden layer 3
model.add(keras.layers.Dense(2,  activation='softmax'))     # output layer (2 classes)

model.summary()

# ── Step 4: Compile the model ──────────
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Step 5: Fit the model ─────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ── Step 6: Evaluate on test data ────────
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


# ── Step 7: Save the model ─────────
model.save("tic-tac-toe.model.keras")
print("\nModel saved to tic-tac-toe.model.keras")




Training samples : 766
Test samples     : 192


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,282 (12.82 KB)

 Trainable params: 3,282 (12.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6343 - loss: 0.6269 - val_accuracy: 0.7143 - val_loss: 0.5306
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6734 - loss: 0.5762 - val_accuracy: 0.8052 - val_loss: 0.4960
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7402 - loss: 0.5423 - val_accuracy: 0.8182 - val_loss: 0.4603
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7562 - loss: 0.5138 - val_accuracy: 0.8182 - val_loss: 0.4440
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7823 - loss: 0.4830 - val_accuracy: 0.8052 - val_loss: 0.4409
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8142 - loss: 0.4517 - val_accuracy: 0.8571 - val_loss: 0.3889
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8316 - loss: 0.4118 - val_accuracy: 0.8571 - val_loss: 0.3701
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8505 - loss: 0.3693 - val_accuracy: 0.8571 - val_loss

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras

# ── Reproduce the same preprocessing & split ─────────────────────────────────
df = pd.read_csv("tic-tac-toe.csv")

mapping = {'x': 1, 'o': -1, 'b': 0}
feature_cols = [c for c in df.columns if c != 'class']
for col in feature_cols:
    df[col] = df[col].map(mapping)
df['class'] = df['class'].astype(int)

X = df[feature_cols]
y = df['class']

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

_, X_test, _, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ── Load the saved model ──────────────────────────────────────────────────────
model = keras.models.load_model("tic-tac-toe.model.keras")
print("Model loaded successfully.\n")

# ── Pick 10 random rows from the test set ────────────────────────────────────
np.random.seed(0)
sample_indices = np.random.choice(X_test.index, size=10, replace=False)

X_sample = X_test.loc[sample_indices]
y_sample = y_test.loc[sample_indices]

# ── Make predictions ──────────────────────────────────────────────────────────
probabilities = model.predict(X_sample, verbose=0)   # shape: (10, 2)
predicted_classes = np.argmax(probabilities, axis=1)  # 0 or 1

# ── Display results — single summary on test data ───────────────────────
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("=" * 40)
print("       TEST SET PERFORMANCE")
print("=" * 40)
print(f"  Loss     : {test_loss:.4f}  {'✓ < 0.1' if test_loss < 0.1 else '✗ >= 0.1'}")
print(f"  Accuracy : {test_accuracy:.4f}  {'✓ > 0.95' if test_accuracy > 0.95 else '✗ <= 0.95'}")
print("=" * 40)

Model loaded successfully.

       TEST SET PERFORMANCE
  Loss     : 0.0376  ✓ < 0.1
  Accuracy : 0.9896  ✓ > 0.95


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [10]:
from keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras



#
# Adding epochs and dropout to prevent overfitting
#

# ── Step 1: Split into training and test sets ────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

# ── Step 2: Create a Sequential model ────────────
model = keras.Sequential()

# ── Step 3: Add layers ──────────
model.add(keras.layers.Input(shape=(X_train.shape[1],)))    # input layer
model.add(keras.layers.Dense(64, activation='relu'))        # hidden layer 1
model.add(keras.layers.Dropout(0.3))                        # dropout
model.add(keras.layers.Dense(32, activation='relu'))        # hidden layer 2
model.add(keras.layers.Dense(16, activation='relu'))        # hidden layer 3
model.add(keras.layers.Dense(2,  activation='softmax'))     # output layer (2 classes)

model.summary()

# ── Step 4: Compile the model ──────────
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Step 5: Fit the model ─────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ── Step 6: Evaluate on test data ────────
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


# ── Step 7: Save the model ─────────
model.save("tic-tac-toe2.model.keras")
print("\nModel saved to tic-tac-toe2.model.keras")


##########



# ── Reproduce the same preprocessing & split ─────────────────────────────────
df = pd.read_csv("tic-tac-toe.csv")

mapping = {'x': 1, 'o': -1, 'b': 0}
feature_cols = [c for c in df.columns if c != 'class']
for col in feature_cols:
    df[col] = df[col].map(mapping)
df['class'] = df['class'].astype(int)

X = df[feature_cols]
y = df['class']

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

_, X_test, _, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ── Load the saved model ──────────────────────────────────────────────────────
model = keras.models.load_model("tic-tac-toe2.model.keras")
print("Model 2 loaded successfully.\n")

# ── Pick 10 random rows from the test set ────────────────────────────────────
np.random.seed(0)
sample_indices = np.random.choice(X_test.index, size=10, replace=False)

X_sample = X_test.loc[sample_indices]
y_sample = y_test.loc[sample_indices]

# ── Make predictions ──────────────────────────────────────────────────────────
probabilities = model.predict(X_sample, verbose=0)   # shape: (10, 2)
predicted_classes = np.argmax(probabilities, axis=1)  # 0 or 1

# ── Display results — single summary on test data ───────────────────────
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("=" * 40)
print("      2nd TEST SET PERFORMANCE")
print("=" * 40)
print(f"  Loss     : {test_loss:.4f}  {'✓ < 0.1' if test_loss < 0.1 else '✗ >= 0.1'}")
print(f"  Accuracy : {test_accuracy:.4f}  {'✓ > 0.95' if test_accuracy > 0.95 else '✗ <= 0.95'}")
print("=" * 40)


Training samples : 766
Test samples     : 192


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,282 (12.82 KB)

 Trainable params: 3,282 (12.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6473 - loss: 0.6458 - val_accuracy: 0.7143 - val_loss: 0.5873
Epoch 2/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.6188 - val_accuracy: 0.7273 - val_loss: 0.5631
Epoch 3/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6575 - loss: 0.6079 - val_accuracy: 0.7922 - val_loss: 0.5373
Epoch 4/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6894 - loss: 0.5877 - val_accuracy: 0.7922 - val_loss: 0.5267
Epoch 5/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6938 - loss: 0.5783 - val_accuracy: 0.7922 - val_loss: 0.5122
Epoch 6/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7141 - loss: 0.5598 - val_accuracy: 0.7792 - val_loss: 0.5039
Epoch 7/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7358 - loss: 0.5507 - val_accuracy: 0.7922 - val_loss: 0.4952
Epoch 8/80
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7344 - loss: 0.5536 - val_accuracy: 0.7922 - val_loss

**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
# Adding epochs and dropout did not improve the Loss and Accuracy